# ⚡ AURA FLUX.1 — High-End Unrestricted AI Studio for Google Colab

This notebook deploys the **AURA FLUX & SDXL** image generation engine with a luxury dark glassmorphism Web UI directly inside Google Colab.

### Features
- **Unrestricted Generation** — Clean diffusers inference without artificial safety filtering.
- **A100/T4 Auto-Optimization** — Automatic BF16/FP16 precision and CPU offloading for any GPU.
- **Multi-Model Support** — FLUX.1-schnell, FLUX.1-dev, Qwen-Image-2.1, SDXL 1.0, SD 3.5 Medium.
- **Dynamic LoRA Loader** — Combine up to 3 HuggingFace LoRA checkpoints on-the-fly.
- **T2I & I2I Support** — Full Text-to-Image and Image-to-Image capabilities.
- **Public HTTPS Link** — Instant tunnel generation via ngrok.

### Requirements
- **Runtime → Change runtime type → GPU** (A100 recommended, T4 supported with CPU offloading)
- For gated models (FLUX.1-dev), set your HuggingFace token in the secrets panel or in the cell below.

In [ ]:
# ============================================================
# STEP 1: GPU Detection & Validation
# ============================================================
import torch

if not torch.cuda.is_available():
    print("❌ ERROR: No GPU detected!")
    print("   Go to Runtime → Change runtime type → Select GPU (T4 or A100)")
    raise SystemExit("GPU required. Change runtime type and re-run.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
compute_cap = torch.cuda.get_device_capability(0)

print(f"✅ GPU Detected: {gpu_name}")
print(f"✅ VRAM: {vram_gb} GB")
print(f"✅ Compute Capability: {compute_cap[0]}.{compute_cap[1]}")

if compute_cap[0] >= 8:
    print(f"✅ BF16 Precision: Supported (A100/H100 Optimized)")
else:
    print(f"⚠️ BF16 NOT supported on this GPU — will use FP16")

if vram_gb < 15:
    print(f"⚠️ Low VRAM ({vram_gb}GB) — FLUX models will use CPU offloading (slower)")
    print(f"   Consider using SDXL or SD 3.5 Medium for best performance on this GPU.")

In [ ]:
# ============================================================
# STEP 2: Clone Repository & Install Dependencies
# ============================================================
# NOTE: Replace with your actual GitHub/GitLab repo URL
# If you uploaded files manually to Colab, skip the git clone line.

import os

REPO_URL = "https://github.com/krishnagopalmishra1-we/Flux-Collab"
PROJECT_DIR = "/content/aura-flux-studio"

if REPO_URL:
    if os.path.exists(PROJECT_DIR):
        print('Updating repository...')
        os.chdir(PROJECT_DIR)
        !git pull
    else:
        !git clone {REPO_URL} {PROJECT_DIR}
        os.chdir(PROJECT_DIR)
else:
    # Manual upload mode: upload server.py, engine.py, static/, templates/ to /content/
    from google.colab import files as colab_files
    print("📁 No REPO_URL set. Upload your project files manually:")
    print("   Required: server.py, engine.py, static/style.css, static/app.js, templates/index.html")
    print("   Or set REPO_URL above and re-run this cell.")
    # Alternatively, mount Google Drive:
    # from google.colab import drive
    # drive.mount('/content/drive')
    # os.chdir('/content/drive/MyDrive/aura-flux-studio')

# Install only what Colab doesn't have pre-installed
!pip install -q diffusers>=0.30.0 transformers accelerate safetensors peft sentencepiece huggingface-hub fastapi uvicorn python-multipart pyngrok

print("\n✅ Dependencies installed successfully.")

In [ ]:
# ============================================================
# STEP 3: (Optional) Set HuggingFace Token for Gated Models
# ============================================================
# Required for: FLUX.1-dev, Qwen-Image-2.1
# Not required for: FLUX.1-schnell, SDXL 1.0, SD 3.5 Medium

import os

# Option A: Set directly (replace with your token)
# os.environ['HF_TOKEN'] = 'hf_your_token_here'

# Option B: Use Colab Secrets (recommended - set HF_TOKEN in the 🔑 panel)
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print("✅ HF_TOKEN loaded from Colab Secrets.")
    else:
        print("ℹ️ No HF_TOKEN found. Only ungated models (FLUX.1-schnell, SDXL) will work.")
except Exception:
    print("ℹ️ Colab Secrets not available. Set HF_TOKEN manually if needed.")

In [ ]:
# ============================================================
# STEP 4: Launch AURA FLUX Studio
# ============================================================
import subprocess
import time
from google.colab import output

# Ensure output directories exist
os.makedirs('static', exist_ok=True)
os.makedirs('templates', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

# Verify critical files exist
for f in ['server.py', 'engine.py']:
    if not os.path.exists(f):
        raise FileNotFoundError(f"❌ {f} not found! Upload project files or set REPO_URL in Step 2.")

# Kill any existing server on port 7860 to avoid Address Already in Use errors
os.system("fuser -k 7860/tcp")
# Start FastAPI server in background
server_process = subprocess.Popen(
    ['python', 'server.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

# Wait for server to boot
print("⏳ Starting AURA FLUX Studio server...")
time.sleep(5)

if server_process.poll() is not None:
    # Server crashed on startup
    out = server_process.stdout.read().decode()
    print(f"❌ Server failed to start:\n{out}")
    raise RuntimeError("Server startup failed.")

print("\n" + "=" * 60)
print("  ✨ AURA FLUX Studio is running on port 7860!")
print("=" * 60)

# Display inside Colab notebook via iframe
output.serve_kernel_port_as_iframe(7860, height=900)

# Provide a direct proxy link (bypasses iframe cookie issues)
proxy_url = output.eval_js(f'google.colab.kernel.proxyPort(7860)')
print('\n' + '=' * 60)
print(f'🌍 DIRECT COLAB LINK (Click this if iframe is broken):\n{proxy_url}')
print('=' * 60 + '\n')

print("\n💡 Tip: For external access, run the next cell to create a public ngrok tunnel.")

In [ ]:
# ============================================================
# STEP 5: (Optional) Create Public ngrok Tunnel
# ============================================================
# Set your ngrok auth token at: https://dashboard.ngrok.com/auth

NGROK_AUTH_TOKEN = ""  # Paste your ngrok auth token here

if NGROK_AUTH_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(7860)
    print(f"\n🌍 Public URL: {public_url}")
    print(f"   Share this link to access your studio from any device.")
else:
    print("ℹ️ Set NGROK_AUTH_TOKEN above to create a public URL.")
    print("   Get a free token at: https://dashboard.ngrok.com/auth")